In [2]:
# ============================================================================
# 3_tables.ipynb  —  All statistics and tables (OFFLINE, no API)
# ----------------------------------------------------------------------------
# MERGES former 04 (divergence + 2x2 ablation + agreement), the analysis halves
# of 03c (production-metric divergence) / 03d (judge-model table) / 06 (construct
# table) / 03b (inter-judge), and 08 (bootstrap). Reads only persisted parquets.
#
# PATCH (second-round review): pairing is now collapsed to one value per unique
# case_id, fixing the n inflation (114 override instances were cartesian-expanded
# to 144 pseudo-pairs because case_id = cik_year is not unique across agencies).
#
# INPUT   data/scored.parquet, data/judge_models.parquet,
#         data/construct_scored.parquet, data/interjudge.parquet
# OUTPUT  results/tables/{tab_divergence, tab_divergence_rag, tab_ablation_2x2,
#         tab_judge_models, tab_construct_validity, tab_interjudge, tab_bootstrap_p4}
# ============================================================================
import sys, os, json, warnings
from pathlib import Path
import numpy as np
import pandas as pd

_HERE = Path.cwd()
_ROOT = _HERE if (_HERE / "src").exists() else _HERE.parent
sys.path.insert(0, str(_ROOT / "src"))
from common import (PROJECT_ROOT, DATA_DIR, RESULTS_DIR, FIGURES_DIR, TABLES_DIR,
                    load_env, set_grayscale_style, save_fig, save_table,
                    require_input, RANDOM_SEED)
import matplotlib.pyplot as plt
import seaborn as sns
set_grayscale_style()
rng = np.random.default_rng(RANDOM_SEED)
warnings.filterwarnings("ignore")

from scipy.stats import wilcoxon, spearmanr
from statsmodels.stats.multitest import multipletests
from sklearn.metrics import cohen_kappa_score

s = pd.read_parquet(require_input(DATA_DIR / "scored.parquet", hint="Run notebook 2 first."))
s["faith5"] = s["faithfulness"] * 5.0
if "faithfulness_rag" in s.columns:
    s["rag5"] = s["faithfulness_rag"] * 5.0
head = s[(s.judge_cond == "informed") & (s.condition == "gap_given")].copy()

def _paired_series(frame, col, pert_code):
    # Collapse to ONE value per unique case_id before pairing. case_id = cik_year
    # is NOT unique when a firm carries several agency ratings in the same year, and
    # a raw set_index join then cartesian-expands those duplicates, inflating the
    # paired n (e.g. 114 override instances -> 144 pseudo-pairs). Averaging duplicate
    # instances per case_id makes each firm-year contribute exactly once, so n is the
    # number of unique override firm-years and the paired tests stay independent.
    p0 = frame[frame.pert == "P0"].groupby("case_id")[col].mean()
    pk = frame[(frame.pert == pert_code) & (frame.k.isin([0, 3]))].groupby("case_id")[col].mean()
    return p0.rename("p0").to_frame().join(pk.rename("pk"), how="inner").dropna()

# ---- (1) divergence: judge drop vs PROXY metric drop -------------------------
def divergence_row(pert_code, metric_col):
    j = _paired_series(head, "judge_qual", pert_code); jdrop = j.p0 - j.pk
    m = _paired_series(head, metric_col, pert_code);  mdrop = m.p0 - m.pk
    div = jdrop.reindex(mdrop.index).fillna(0) - mdrop
    try: pj = wilcoxon(j.p0, j.pk).pvalue
    except Exception: pj = np.nan
    try: pdv = wilcoxon(div).pvalue
    except Exception: pdv = np.nan
    dz = jdrop.mean() / (jdrop.std(ddof=1) + 1e-9)
    return dict(pert=pert_code, judge_drop=round(float(jdrop.mean()), 3),
                metric_drop=round(float(mdrop.mean()), 3),
                divergence=round(float(div.mean()), 3), d_z=round(float(dz), 3),
                p_judge=pj, p_div=pdv)

def holm(tab):
    mask = tab["p_div"].notna()
    tab.loc[mask, "p_div_holm"] = multipletests(tab.loc[mask, "p_div"], method="holm")[1]
    return tab

divt = holm(pd.DataFrame([divergence_row(p, "faith5") for p in ["P1", "P2", "P3", "P4", "P5"]]))
save_table(divt, "tab_divergence")
print("== tab_divergence (proxy metric) =="); print(divt.to_string(index=False))
print("paired n (unique override firm-years):",
      len(_paired_series(head, "judge_qual", "P4")))

# ---- (2) divergence under PRODUCTION faithfulness ----------------------------
if "rag5" in s.columns and head["rag5"].notna().any():
    ragt = holm(pd.DataFrame([divergence_row(p, "rag5") for p in ["P1", "P2", "P3", "P4", "P5"]]))
    ragt = ragt.rename(columns={"metric_drop": "rag_metric_drop"})
    save_table(ragt, "tab_divergence_rag")
    print("\n== tab_divergence_rag (production metric) =="); print(ragt.to_string(index=False))

# ---- (3) 2x2 circularity ablation (judge drop on P2, P4 per cell) ------------
def cell_drop(gen_cond, judge_cond, pert):
    d = s[(s.condition == gen_cond) & (s.judge_cond == judge_cond)]
    pr = _paired_series(d, "judge_qual", pert)
    return round(float((pr.p0 - pr.pk).mean()), 3) if len(pr) else np.nan
abl = [{"generation": g, "judge": j, "P2_drop": cell_drop(g, j, "P2"), "P4_drop": cell_drop(g, j, "P4")}
       for g in ("gap_given", "gap_blind") for j in ("informed", "blind")]
ablt = pd.DataFrame(abl); save_table(ablt, "tab_ablation_2x2")
print("\n== tab_ablation_2x2 =="); print(ablt.to_string(index=False))

# ---- (4) judge-model robustness ---------------------------------------------
jm_path = DATA_DIR / "judge_models.parquet"
if jm_path.exists():
    jm = pd.read_parquet(jm_path)
    def jm_stats(model, pert_code):
        d = jm[jm.judge_model == model]
        pr = _paired_series(d.assign(k=0), "judge_qual", pert_code)
        drop = pr.p0 - pr.pk
        if len(drop) < 3: return dict(judge_model=model, pert=pert_code, n=len(drop), judge_drop=np.nan, d_z=np.nan, p=np.nan)
        try: pv = wilcoxon(pr.p0, pr.pk).pvalue
        except Exception: pv = np.nan
        return dict(judge_model=model, pert=pert_code, n=len(drop),
                    judge_drop=round(float(drop.mean()), 3),
                    d_z=round(float(drop.mean() / (drop.std(ddof=1) + 1e-9)), 3), p=pv)
    jmt = pd.DataFrame([jm_stats(m, p) for m in jm.judge_model.unique() for p in ["P3", "P4"]])
    save_table(jmt, "tab_judge_models"); print("\n== tab_judge_models =="); print(jmt.to_string(index=False))

# ---- (5) construct validity --------------------------------------------------
cs_path = DATA_DIR / "construct_scored.parquet"
if cs_path.exists():
    cs = pd.read_parquet(cs_path)
    def cs_stats(pert_code, grade_view):
        d = cs[cs.grade_view == grade_view]
        pr = _paired_series(d.assign(k=0), "judge_qual", pert_code)
        drop = pr.p0 - pr.pk
        if len(drop) < 3: return dict(pert=pert_code, grade_view=grade_view, n=len(drop), judge_drop=np.nan, d_z=np.nan, p=np.nan)
        try: pv = wilcoxon(pr.p0, pr.pk).pvalue
        except Exception: pv = np.nan
        return dict(pert=pert_code, grade_view=grade_view, n=len(drop),
                    judge_drop=round(float(drop.mean()), 3),
                    d_z=round(float(drop.mean() / (drop.std(ddof=1) + 1e-9)), 3), p=pv)
    cst = pd.DataFrame([cs_stats(p, g) for p in ["P1", "P4", "P4b"] for g in ["shown", "hidden"]])
    save_table(cst, "tab_construct_validity"); print("\n== tab_construct_validity =="); print(cst.to_string(index=False))

# ---- (6) inter-judge agreement ----------------------------------------------
ij_path = DATA_DIR / "interjudge.parquet"
if ij_path.exists():
    ij = pd.read_parquet(ij_path).dropna(subset=["judge_qual", "judge_qual_secondary"])
    p = ij["judge_qual"].round().astype(int).clip(1, 5)
    q = ij["judge_qual_secondary"].round().astype(int).clip(1, 5)
    rho, rho_p = spearmanr(ij["judge_qual"], ij["judge_qual_secondary"])
    kappa = cohen_kappa_score(p, q, weights="quadratic", labels=[1, 2, 3, 4, 5])
    ijt = pd.DataFrame({"metric": ["n_pairs", "spearman_rho", "spearman_p", "quadratic_weighted_kappa"],
                        "value": [len(ij), round(float(rho), 3), round(float(rho_p), 4), round(float(kappa), 3)]})
    save_table(ijt, "tab_interjudge"); print("\n== tab_interjudge =="); print(ijt.to_string(index=False))

# ---- (7) bootstrap CIs + permutation test for P4 (and P3 contrast) ----------
N_BOOT, N_PERM = 10000, 10000
def dz(x): x = np.asarray(x, float); return x.mean() / (x.std(ddof=1) + 1e-9)
def paired_div(pert_code):
    j = _paired_series(head, "judge_qual", pert_code); m = _paired_series(head, "faith5", pert_code)
    df = pd.DataFrame({"jdrop": j.p0 - j.pk, "mdrop": m.p0 - m.pk}).dropna()
    df["div"] = df["jdrop"] - df["mdrop"]; return df
def boot_rows(pert_code):
    df = paired_div(pert_code); n = len(df)
    jd, md, dv, dzs = [], [], [], []
    for _ in range(N_BOOT):
        b = df.iloc[rng.integers(0, n, n)]
        jd.append(b["jdrop"].mean()); md.append(b["mdrop"].mean())
        dv.append(b["div"].mean()); dzs.append(dz(b["jdrop"]))
    ci = lambda a: (round(float(np.percentile(a, 2.5)), 3), round(float(np.percentile(a, 97.5)), 3))
    obs = df["div"].mean(); v = df["div"].values
    perm = (sum(abs((v * rng.choice([-1.0, 1.0], len(v))).mean()) >= abs(obs) for _ in range(N_PERM)) + 1) / (N_PERM + 1)
    return dict(pert=pert_code, n=n, judge_drop_mean=round(float(df["jdrop"].mean()), 3), judge_drop_CI=ci(jd),
                metric_drop_mean=round(float(df["mdrop"].mean()), 3), metric_drop_CI=ci(md),
                divergence_mean=round(float(df["div"].mean()), 3), divergence_CI=ci(dv),
                d_z=round(float(dz(df["jdrop"])), 3), d_z_CI=ci(dzs), perm_p_divergence=round(float(perm), 4))
boot = pd.DataFrame([boot_rows("P4"), boot_rows("P3")])
save_table(boot, "tab_bootstrap_p4"); print("\n== tab_bootstrap_p4 =="); print(boot.to_string(index=False))
print("\n[done] all tables written.")

[saved] results\tables\tab_divergence.csv
[saved] results\tables\tab_divergence.tex
== tab_divergence (proxy metric) ==
pert  judge_drop  metric_drop  divergence   d_z      p_judge        p_div   p_div_holm
  P1       0.424       -0.013       0.438 0.483 1.351619e-05 4.421295e-03 1.326388e-02
  P2       0.702        0.615       0.087 0.667 5.082444e-08 6.071892e-01 1.000000e+00
  P3       0.192        1.083      -0.891 0.278 4.276334e-03 1.400788e-04 5.603153e-04
  P4       1.793        0.300       1.493 1.244 1.612493e-13 1.633999e-08 8.169994e-08
  P5       0.348        0.466      -0.118 0.573 8.016947e-07 9.734870e-01 1.000000e+00
paired n (unique override firm-years): 99
[saved] results\tables\tab_divergence_rag.csv
[saved] results\tables\tab_divergence_rag.tex

== tab_divergence_rag (production metric) ==
pert  judge_drop  rag_metric_drop  divergence   d_z      p_judge        p_div   p_div_holm
  P1       0.424            0.007       0.418 0.483 1.351619e-05 1.160695e-05 2.321390e